# 3 · Check and submit

Notebook 2 chose a model on one split of the training images. Before submitting it, we
check four things, each of which could still sink it:

1. **Is it really as good as the leader?** On images that played no part in any choice.
2. **Does it cost what we think?** Counted by the scorer itself, not by our lab.
3. **Does it train in time?** On a 4-core CPU like GitHub's, the limit is 10 minutes.
4. **Where does it fail?** The mistakes it makes, next to the leader's.

Then we write `model.py` and submit.

## The choice

The cheapest setting that tied the leader in experiment 11, decided *before* any of the
checks below were run, and written down in `plans.py` as `FINAL`:

In [ ]:
import lab
import matplotlib.pyplot as plt
import pandas as pd
from plans import FINAL

final, leader = lab.leader(**FINAL), lab.leader()
rows = []
for name in ("widths", "optimizer", "lr", "weight_decay", "batch_size", "epochs", "low_res", "tta"):
    rows.append({"setting": name, "leader": getattr(leader, name), "ours": getattr(final, name)})
display(pd.DataFrame(rows).set_index("setting"))
for name, cfg in (("leader", leader), ("ours", final)):
    print(
        f"{name:6}: training {lab.train_flops(cfg, 21600) / 1e12:5.2f} TFLOP,"
        f" answering {lab.predict_flops(cfg) / 1e6:6.1f} MFLOP per image,"
        f" {lab.parameters(cfg):,} parameters"
    )

Everything else is the leader's: the same shape of network, the same turning and
mirroring, the same one-cycle schedule.

## 1 · Is it as good as the leader, on fresh images?

Every choice in notebook 2 was made on one split of the training images (`seed=1`).
A setting that won there is a little lucky on exactly those images: with dozens of
comparisons, some winners won by luck. So we check on two other splits (`seed=7`, the
one `uv run score` uses, and `seed=3`): new check images, and new training images too.
The leader and our choice, 3 runs each on each split, side by side. Nothing is chosen
from these runs. We also ran 45 and 75 epochs, to see the trade-off on fresh images.

In [ ]:
fresh = lab.load("12-confirm")
table = fresh.pivot_table(index="label", columns="split", values="accuracy", aggfunc="mean") * 100
table["both splits"] = fresh.groupby("label")["accuracy"].mean() * 100
table["train TFLOP"] = fresh.groupby("label")["train_flops_full"].mean() / 1e12
table["answer MFLOP"] = fresh.groupby("label")["predict_flops"].mean() / 1e6
order = ["leader", "45 epochs", "final: 60 epochs", "75 epochs"]
display(table.loc[order].round(2))
for label in order[1:]:
    print(lab.compare(fresh, "leader", label))

**Result.** On fresh images, our model scores 96.9% and the leader 97.1%: a difference
of 0.24 points, within luck (± 0.16, so 1.5 standard errors). On split 7 we are level
(97.2 vs 97.1); on split 3 we are half a point behind. **A tie, at a quarter of the
training compute and a seventh of the compute per image.**

Two things worth noticing:

- **Every model scores higher here than on the `seed=1` split** (the leader 97.1
  instead of 96.9). Some splits are simply easier. This is why we compare models on the
  *same* images, never a number from one split with a number from another.
- **The trade-off held on fresh images**: 45 epochs is clearly worse than the leader
  (−0.65, real), 75 epochs a little better (+0.16, luck). Had we wanted to *beat* the
  leader rather than match it, 75 epochs (a third of its compute) would be the choice.
  We decided on 60 before running these, and a decision made before seeing the result
  is the only one these runs can confirm.

## 2 · Did every part earn its place?

Each experiment in notebook 2 added one thing. An *ablation* looks back from the end:
take the finished model, undo one choice, keep everything else, and see what it was
worth. Where undoing a choice changes the cost per epoch, the epochs change so that
every row costs the same 6.5 TFLOP. We did the same for the leader, to see which of its
parts were worth their complication. All on the `seed=1` split, 3 runs each.

In [ ]:
def ablation(name, reference):
    t = lab.load(name)
    s = lab.summary(t)[["accuracy", "se", "train_tflop", "predict_mflop"]]
    s["training accuracy"] = (t.groupby("label", sort=False)["train_accuracy"].mean() * 100).round(
        2
    )
    s["vs " + reference] = [
        "" if label == reference else lab.compare(t, reference, label).split(": ")[1]
        for label in s.index
    ]
    return s


display(ablation("14-final-ablation", "final"))

Our model, one choice undone at a time:

- **The leader's width** (with the 16 epochs that fit): −2.0 points. The central
  decision of notebook 2: spend compute on epochs, not on width.
- **No turning and mirroring in training**: −1.1. Look at the training accuracy: 98.6%
  without them, against 96.1% with them. Without new orientations to learn from, a long
  training starts to memorize. (At 16 epochs, in experiment 2, it did not matter yet.)
- **All epochs at full size** (26 fit instead of 60): −0.7.
- **Batch 128**: −0.55, but noisy (± 0.34): probably real, not proven here.
- **AdamW instead of SGD**: −0.16, within luck. Experiment 9 found +0.57 for SGD at 45
  epochs; at 60 epochs, with the same 3 seeds, the difference is too small to see. We
  keep SGD (it is no worse), but the honest summary is that the optimizer matters less
  than it looked. One experiment is a hint, not a law: a gain found in one setting may
  shrink in another.

In [ ]:
display(ablation("13-leader-ablation", "leader"))
views = pd.DataFrame(list(lab.load("0-leader")["accuracy_by_views"])).astype(float).mean() * 100
print("the leader answering with 1, 2, 4, 8 orientations:", views.round(2).to_dict())

The leader, one part removed at a time:

- **Earned their place**: turning and mirroring in training (+1.3; without them it
  memorizes: 99.9% training accuracy), the max pool after the first layer (+1.0: moving
  4 pixels at a time instead skips pixels and loses texture), and the warm-up at the
  start of the one-cycle schedule (+0.65).
- **Did not**: the residual shortcuts (−0.08, luck: with only 3 blocks the network is
  shallow enough to train without them), weight decay (−0.18, luck), and orientations 5
  to 8 when answering (+0.02 for half of its compute per image).

Shortcuts are a good example of a complication copied from bigger networks, where they
are essential (a 50-layer network does not train without them), into a small one where
they are not. We kept them in our model: we did not test removing them there, and an
untested change is not a decision.

## The ladder: what each step bought

All on the `seed=1` split, each a mean of 3 to 6 runs, from the experiments of
notebook 2. Each rung keeps everything above it and changes one thing.

In [ ]:
def mean_of(experiment, label):
    t = lab.load(experiment)
    rows = t[t["label"] == label]
    return 100 * rows["accuracy"].mean(), rows["train_flops_full"].mean() / 1e12


ladder = pd.DataFrame(
    [
        ("the leader", *mean_of("0-leader", "leader")),
        ("half width, 16 epochs (exp 1)", *mean_of("1-size-and-time", "width 0.5 x 16 epochs")),
        (
            "+ batch 64, learning rate 0.024 (exp 5)",
            *mean_of("5-batch-and-lr", "batch 64, lr 0.024"),
        ),
        ("+ 3/4 of 45 epochs at 32x32 (exp 7)", *mean_of("8-regularize", "candidate")),
        ("+ SGD instead of AdamW (exp 9, 11)", *mean_of("11-sgd-budget", "45 epochs")),
        ("+ 60 epochs instead of 45 (exp 11)", *mean_of("11-sgd-budget", "60 epochs")),
    ],
    columns=["step", "accuracy", "train TFLOP"],
).set_index("step")
display(ladder.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ladder["train TFLOP"], ladder["accuracy"], "o-", color="tab:blue")
for i, (step, row) in enumerate(ladder.iterrows()):
    ax.annotate(
        f"{i}", (row["train TFLOP"], row["accuracy"]), textcoords="offset points", xytext=(6, -4)
    )
ax.axvline(26.22 / 5, color="gray", linestyle="--")
ax.text(26.22 / 5 * 1.03, 93.5, "1/5 of the leader", color="gray")
ax.set_xscale("log")
ax.set_xlabel("training compute (TFLOP, log scale)")
ax.set_ylabel("accuracy, seed=1 split (%)")
ax.set_title("From the leader (0) to our model (5)")
plt.show()

Halving the width and training for 16 epochs cut the compute by 6.6 and cost 3.5
points. Four changes that cost no compute (smaller batches, a higher learning rate,
small images first, SGD) won almost all of it back, and a few more epochs closed the
gap. Not one of them is a new idea; what made them work is that each was *measured*,
at equal compute, with enough runs to tell it from luck.

## 3 · Write the submission

GitHub only ever sees `model.py`, and it always uses its defaults. This cell writes it:
the defaults are the choice above, and each one names the experiment that chose it.
The other settings stay in the file so that you can repeat any experiment.

In [ ]:
%%writefile model.py
"""Demo 2: as accurate as the top of the leaderboard, for a quarter of its compute.

A residual network half as wide as the leader's (BuzinD), trained with SGD for 60 epochs,
45 of them on half-size images, and answering from 4 orientations of each image.
How it was built, and why each setting has its value: the notebooks in this folder.
Notebook 3's submission cell writes this file, so make your changes there.

GitHub calls train(train_set, log), which uses the defaults in Config.
The notebooks call train(train_set, log, Config(epochs=8)) to try other values.
"""

import copy
import math
from collections.abc import Callable
from dataclasses import dataclass

import torch
from torch import nn
from torch.nn import functional as F


@dataclass(frozen=True)  # frozen: a setting cannot be changed by accident
class Config:
    # The defaults are our choice; "exp N" is the experiment in notebook 2 that chose it.
    # The network: the leader's shape (exp 3), at half its width (exp 1, 4, 7)
    stem: str = "conv"  # first layer: "conv" (3x3 windows) or "patch" (cut into squares)
    stem_stride: int = 2  # "conv": move this many pixels at a time; "patch": square size
    stem_pool: bool = True  # "conv" only: halve the image again with a max pool
    block: str = "basic"  # "basic": two 3x3 convolutions; "inverted": cheap depthwise ones
    widths: tuple[int, ...] = (12, 24, 48)  # channels in each stage
    depths: tuple[int, ...] = (1, 1, 1)  # blocks per stage; each stage after the first halves
    expand: int = 4  # "inverted" only: how much wider the inside of a block is
    zero_init: bool = False  # start each block as "do nothing" (its last scale at 0)
    shortcut: bool = True  # add each block's input to its output (a residual network)
    # Training
    optimizer: str = "sgd"  # "adamw", or "sgd" with Nesterov momentum 0.9 (exp 9)
    epochs: int = 60  # passes over all training images (exp 11)
    batch_size: int = 64  # images per learning step (exp 2, 5)
    lr: float = 0.1  # learning rate: the largest step size (exp 9)
    weight_decay: float = 5e-4  # pull weights toward 0 (exp 9)
    pct_start: float = 0.2  # share of training spent raising the learning rate
    label_smoothing: float = 0.0  # aim for 0.9 instead of 1 on the right class (exp 8: no)
    mixup: float = 0.0  # blend pairs of images and their labels (exp 8: no)
    ema: float = 0.0  # keep a running average of the weights, and return that (exp 8: no)
    augment: bool = True  # randomly turn and mirror images
    low_res: float = 0.75  # share of epochs trained on half-size (32x32) images (exp 6, 7)
    # Answering
    tta: int = 4  # look at the image in 1, 2, 4 or 8 orientations, average the answers (exp 0)
    # Watching
    valid_images: int = 0  # kept out of training, only to draw the validation curve
    device: str = "cpu"  # notebooks only: "cuda" or "mps" trains on a GPU (GitHub has none)
    watch: Callable | None = None  # notebooks only: gets (model, optimizer) to look inside


# ---------------------------------------------------------------- the network


class Basic(nn.Module):
    """Two 3x3 convolutions and a shortcut around them (a ResNet "basic block")."""

    def __init__(self, c_in: int, c_out: int, stride: int, cfg: Config) -> None:
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(),
            nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False),
            nn.BatchNorm2d(c_out),
        )
        self.skip = shortcut(c_in, c_out, stride) if cfg.shortcut else None
        if cfg.zero_init:
            nn.init.zeros_(self.body[-1].weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.body(x)
        return F.relu(out if self.skip is None else out + self.skip(x))


class Inverted(nn.Module):
    """Widen with a 1x1 convolution, look around with a cheap depthwise 3x3, narrow again.

    A depthwise 3x3 convolution looks at each channel on its own: 9 multiplies per
    channel and pixel instead of 9 x channels. The 1x1 convolutions mix the channels.
    This is the block of MobileNetV2.
    """

    def __init__(self, c_in: int, c_out: int, stride: int, cfg: Config) -> None:
        super().__init__()
        wide = c_in * cfg.expand
        self.body = nn.Sequential(
            nn.Conv2d(c_in, wide, 1, bias=False),
            nn.BatchNorm2d(wide),
            nn.ReLU(),
            nn.Conv2d(wide, wide, 3, stride, 1, groups=wide, bias=False),  # depthwise
            nn.BatchNorm2d(wide),
            nn.ReLU(),
            nn.Conv2d(wide, c_out, 1, bias=False),
            nn.BatchNorm2d(c_out),
        )
        self.skip = shortcut(c_in, c_out, stride)
        if cfg.zero_init:
            nn.init.zeros_(self.body[-1].weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.body(x) + self.skip(x)


def shortcut(c_in: int, c_out: int, stride: int) -> nn.Module:
    """The way around a block: nothing to do, or a 1x1 convolution when the shape changes."""
    if stride == 1 and c_in == c_out:
        return nn.Identity()
    return nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))


def make_model(cfg: Config = Config()) -> nn.Sequential:
    w = cfg.widths[0]
    if cfg.stem == "patch":  # cut the image into squares and describe each one
        stem = [nn.Conv2d(3, w, cfg.stem_stride, cfg.stem_stride, bias=False)]
    else:
        stem = [nn.Conv2d(3, w, 3, cfg.stem_stride, 1, bias=False)]
    stem += [nn.BatchNorm2d(w), nn.ReLU()]
    if cfg.stem == "conv" and cfg.stem_pool:
        stem.append(nn.MaxPool2d(2))

    block = {"basic": Basic, "inverted": Inverted}[cfg.block]
    blocks, c_in = [], w
    for stage, (c_out, depth) in enumerate(zip(cfg.widths, cfg.depths, strict=True)):
        for i in range(depth):
            stride = 2 if stage > 0 and i == 0 else 1  # each new stage halves the size
            blocks.append(block(c_in, c_out, stride, cfg))
            c_in = c_out
    return nn.Sequential(
        *stem,
        *blocks,
        nn.AdaptiveAvgPool2d(1),  # average each pattern over the whole image
        nn.Flatten(),
        nn.Linear(c_in, 10),  # one score per class
    )


# The 4 quarter turns of an image, and each of them mirrored: 8 orientations.
# The first 2 (and the first 4) are a group on their own: turning by half (or by a
# quarter) again gives one of them, so averaging over them treats each alike.
def turn(x: torch.Tensor, k: int) -> torch.Tensor:
    """Turn the images x by k quarter turns (k = 0-3), then mirror them if k is 4-7.

    Made of flips and a transpose, not torch.rot90: the scorer's FLOP counter knows those
    move numbers without arithmetic, and would flag rot90 as uncounted work.
    """
    quarter = k % 4
    if quarter == 1:  # a quarter turn: swap rows and columns, then flip upside down
        x = x.transpose(2, 3).flip(2)
    elif quarter == 2:  # a half turn: flip both ways
        x = x.flip(2, 3)
    elif quarter == 3:  # three quarters: swap rows and columns, then flip left-right
        x = x.transpose(2, 3).flip(3)
    return x.flip(3) if k >= 4 else x


ORIENTATIONS = (0, 2, 1, 3, 4, 6, 5, 7)


class Average(nn.Module):
    """Answer for several orientations of each image and average the scores."""

    def __init__(self, core: nn.Module, views: int) -> None:
        super().__init__()
        self.core, self.views = core, views

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.views == 1:
            return self.core(x)
        return sum(self.core(turn(x, k)) for k in ORIENTATIONS[: self.views]) / self.views


# ---------------------------------------------------------------- training


def coin(x: torch.Tensor) -> torch.Tensor:
    """True for about half of the images in the batch x, at random."""
    return torch.rand(len(x), 1, 1, 1, device=x.device) < 0.5


def flip_and_turn(x: torch.Tensor) -> torch.Tensor:
    """Give each image one of its 8 orientations, at random."""
    x = torch.where(coin(x), x.flip(3), x)  # mirror left-right
    x = torch.where(coin(x), x.flip(2), x)  # mirror top-bottom
    x = torch.where(coin(x), x.transpose(2, 3), x)  # swap rows and columns
    return x.contiguous()


def update_average(average: nn.Module, model: nn.Module, decay: float) -> None:
    """Move the averaged weights a little toward the current ones (cheap: no FLOPs)."""
    with torch.no_grad():
        mine = list(average.state_dict().values())
        theirs = list(model.state_dict().values())
        floats = [(a, b) for a, b in zip(mine, theirs, strict=True) if a.is_floating_point()]
        torch._foreach_lerp_([a for a, _ in floats], [b for _, b in floats], 1 - decay)
        for a, b in zip(mine, theirs, strict=True):
            if not a.is_floating_point():
                a.copy_(b)


def train(train_set, log, cfg: Config = Config()) -> nn.Module:
    images, labels = train_set.images.to(cfg.device), train_set.labels.to(cfg.device)

    # Keep a few images out of training, only to draw the validation curve.
    order = torch.randperm(len(images), device=cfg.device)
    valid, fit = order[: cfg.valid_images], order[cfg.valid_images :]
    if cfg.valid_images:
        log.validate_on(images[valid].cpu(), labels[valid].cpu())
    images, labels = images[fit], labels[fit]

    model = make_model(cfg).to(cfg.device)
    average = copy.deepcopy(model) if cfg.ema else None
    if cfg.optimizer == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(), cfg.lr, momentum=0.9, nesterov=True, weight_decay=cfg.weight_decay
        )
    else:
        optimizer = torch.optim.AdamW(model.parameters(), cfg.lr, weight_decay=cfg.weight_decay)
    steps_per_epoch = math.ceil(len(images) / cfg.batch_size)
    # One cycle: the step size grows to lr, then shrinks to almost nothing by the end.
    schedule = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        cfg.lr,
        epochs=cfg.epochs,
        steps_per_epoch=steps_per_epoch,
        pct_start=cfg.pct_start,
    )
    if cfg.watch:
        cfg.watch(model, optimizer)

    step = 0
    for epoch in range(cfg.epochs):
        model.train()
        small = epoch < round(cfg.low_res * cfg.epochs)  # the first epochs on 32x32 images
        total_loss = torch.zeros((), device=cfg.device)
        correct = torch.zeros((), device=cfg.device)
        for batch in torch.randperm(len(images), device=cfg.device).split(cfg.batch_size):
            x = images[batch].float() / 255  # bytes 0-255 become decimals 0-1
            y = labels[batch]
            if cfg.augment:
                x = flip_and_turn(x)
            if small:
                x = F.avg_pool2d(x, 2)  # each 2x2 square of pixels becomes one
            if cfg.mixup:  # each image becomes a blend of itself and another one
                share = torch.distributions.Beta(cfg.mixup, cfg.mixup).sample().item()
                other = torch.randperm(len(x), device=x.device)
                x = share * x + (1 - share) * x[other]
            scores = model(x)
            loss = F.cross_entropy(scores, y, label_smoothing=cfg.label_smoothing)
            if cfg.mixup:  # ...and should answer with the same blend of the two labels
                other_loss = F.cross_entropy(scores, y[other], label_smoothing=cfg.label_smoothing)
                loss = share * loss + (1 - share) * other_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            schedule.step()
            step += 1
            if average is not None:  # average over about the last 1/(1-ema) steps
                update_average(average, model, min(cfg.ema, (1 + step) / (10 + step)))
            total_loss += loss.detach() * len(batch)
            correct += (scores.argmax(1) == y).sum()
        n = len(images)
        final = model if average is None else average
        numbers = {"train_loss": total_loss.item() / n, "train_accuracy": correct.item() / n}
        if cfg.valid_images:  # GitHub checks the model on the validation images later, free
            log(final, **numbers)
        else:  # no validation images: we train on all of them, and log the training curve
            log(**numbers)
    return Average(final, cfg.tta).cpu().eval()

## 4 · Check it the way GitHub will

`uv run score . --flops` trains `model.py` from scratch on this computer's CPU, exactly
as GitHub does (all 4 cores, no GPU), checks it on the `seed=7` split, and then trains
it a second time inside the scorer's own FLOP counter. It answers the three remaining
questions at once: is the compute what our lab counted, does it train in time, and
where does it fail. (About 10 minutes.)

In [ ]:
!uv run score . --flops

**What it says.**

- **Compute.** The scorer counts the training on the 17,280 images it trains on; on all
  21,600, as on GitHub, that is 1.25 times more: **6.5 TFLOP**, the number our lab
  predicted. **16 MFLOP per image.** Nothing went uncounted. (An earlier version turned
  images with `torch.rot90`, which the counter did not know and flagged as possibly
  heavy; it only moves numbers around, so we rewrote it with flips and a transpose,
  which the counter knows are free. A flag like that turns into a `*` on the
  leaderboard: worth avoiding in a submission meant to be compared.)
- **Time.** About 4 minutes on this 4-core computer, well inside GitHub's 10. Most
  epochs are on small images, which also makes them faster.
- **Mistakes.** The same suspects as in the demo: herbaceous vegetation taken for
  permanent crop (fields with similar patterns), and highways and rivers taken for each
  other (long lines across the image). Forest, residential and sea are almost never
  wrong.

## Summary

| | the leader | ours | |
|---|---|---|---|
| accuracy, fresh splits (6 runs each) | 97.1% | 96.9% | a tie (−0.24 ± 0.16) |
| training compute | 26.2 TFLOP | 6.5 TFLOP | 4.0 times less |
| compute per image | 119 MFLOP | 16 MFLOP | 7.3 times less |
| parameters | 174,274 | 44,230 | 3.9 times fewer |

How we got there, in one sentence: **we counted the compute before spending it, and
then spent it on learning steps instead of width.** Half the width, smaller batches, a
higher learning rate, most epochs on small images, and SGD; each measured at equal
compute, with enough runs to tell a real gain from luck.

## Where to go from here

Ideas we did not try, roughly from most to least promising:

1. **The network still underfits** (training accuracy 96.1%, check accuracy 96.8%), and
   accuracy was still rising with epochs when we stopped (75 epochs: 97.3% on fresh
   images, at a third of the leader's compute). Anything that learns more per FLOP
   should help: for example three image sizes instead of two (16, 32, then 64 pixels),
   or a wider last stage, which is cheap because it works on 4x4 pixels.
2. **Drop the residual shortcuts**: they did not earn their place in the leader, and
   removing them saves a little compute. Test it on our model first.
3. **Answer from 2 orientations** instead of 4: half the compute per image (8 MFLOP),
   for about 0.2 points (experiment 0).
4. **Depthwise convolutions** (the MobileNet block, `block="inverted"` in `model.py`)
   do far less arithmetic than full ones, but the FLOP counter charges their backward
   pass as if they were full convolutions, about 30 times what they compute. Under that
   counter they do not pay; if the counter changes, they become the first thing to try.
5. **Luck**: the same settings score up to 0.5 points apart with different seeds. One
   leaderboard score has about ± 0.3 points of luck in it: keep that in mind before
   celebrating (or mourning) a difference of a few tenths.

Every experiment here is a few lines in `plans.py`. Add yours, run it, and add a line to
`NOTES.md`.